# Imports

In [2]:
import numpy as np
import json
from tqdm import tqdm
import os
import shutil
import librosa
import librosa.display
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# Agrouping songs

In [3]:
name_file = "../jsons/distribution_class.json"
with open(name_file, 'r') as file:
    classes = json.load(file)

In [7]:
source_folder = '../ICBHI_final_database'
destination_folder = '../raw_data'

for filename in tqdm(os.listdir(source_folder), desc='Agruping songs'):
    prefix = filename[:3]

    if prefix in classes:
        class_name = classes[prefix]
        
        class_folder = os.path.join(destination_folder, class_name)
        
        os.makedirs(class_folder, exist_ok=True)
        
        source_file = os.path.join(source_folder, filename)
        destination_file = os.path.join(class_folder, filename)
        
        shutil.move(source_file, destination_file)

FileNotFoundError: [WinError 3] O sistema não pode encontrar o caminho especificado: 'ICBHI_final_database'

# Generate Images

In [3]:
DATA_DIR = "../raw_data"
OUTPUT_DIR = "../processed_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_SR = 18000
TARGET_DURATION = 10


def load_and_standardize(path):
    y, sr = librosa.load(path, sr=None)

    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)

    target_len = TARGET_SR * TARGET_DURATION
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]

    y = y / (np.max(np.abs(y)) + 1e-8)
    return y


def extract_features(y, sr):
    stft = librosa.stft(y)
    mag = np.abs(stft)

    return {
        "spec_db": librosa.amplitude_to_db(mag, ref=np.max),
        "mel_db": librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr), ref=np.max),
        "mfcc": librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20),
        "mfcc_delta": librosa.feature.delta(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)),
        "chroma": librosa.feature.chroma_stft(S=mag, sr=sr),
        "contrast": librosa.feature.spectral_contrast(S=mag, sr=sr, fmin=50, n_bands=4),
        "cqt_db": librosa.amplitude_to_db(
            np.abs(librosa.cqt(y, sr=sr, fmin=30, n_bins=48, bins_per_octave=12)),
            ref=np.max
        ),
        "phase": np.angle(stft)
    }


def save_feature_image(feature, sr, out_path, y_axis=None):
    plt.figure(figsize=(6, 4))

    librosa.display.specshow(
        feature,
        sr=sr,
        x_axis='time',
        y_axis=y_axis
    )

    plt.axis('off')
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches='tight', pad_inches=0)
    plt.close()


feature_configs = {
    "spec_db": {"y_axis": "log"},
    "mel_db": {"y_axis": "mel"},
    "mfcc": {"y_axis": None},
    "mfcc_delta": {"y_axis": None},
    "chroma": {"y_axis": "chroma"},
    "contrast": {"y_axis": None},
    "cqt_db": {"y_axis": "cqt_note"},
    "phase": {"y_axis": None},
}



In [4]:

for disease in os.listdir(DATA_DIR):
    disease_path = os.path.join(DATA_DIR, disease)

    if not os.path.isdir(disease_path):
        continue

    print(f"Processando: {disease}")

    for file in os.listdir(disease_path):
        if not file.endswith(".wav"):
            continue

        file_path = os.path.join(disease_path, file)

        try:
            y = load_and_standardize(file_path)
            features = extract_features(y, TARGET_SR)

            base_name = os.path.splitext(file)[0]

            # salvar cada feature
            for feat_name, feat_value in features.items():
                out_dir = os.path.join(OUTPUT_DIR, feat_name, disease)
                os.makedirs(out_dir, exist_ok=True)

                out_file = os.path.join(out_dir, f"{base_name}.png")

                save_feature_image(
                    feat_value,
                    TARGET_SR,
                    out_file,
                    y_axis=feature_configs[feat_name]["y_axis"]
                )

        except Exception as e:
            print(f"Erro em {file_path}: {e}")

Processando: Asthma
Processando: Bronchiectasis
Processando: Bronchiolitis
Processando: COPD
Processando: Healthy
Processando: LRTI
Processando: Pneumonia
Processando: URTI
